#**Midterm Notebook - DS 2002**

Ayan, Holden, Parker, Andrew

#**Clean Up:**

Loading each CSV and both tables in inventory_and_sales.db. Raw dfs seperate from cleaned data, time stamps = eastern time, later converted to UTC

In [1]:
from pathlib import Path
import re
import sqlite3
import pandas as pd

# paths for both colab & vs code
possible_paths = [
    Path.cwd() / "data",  #cwd
    Path.cwd().parent / "data",  # parent direc
    Path("/content/data"),   # colab data path
]

# check if '/content' (colab)
colab_root = Path("/content")
if colab_root.exists():
    fallback_paths = list(colab_root.glob("**/data"))
    possible_paths.extend(fallback_paths)

DATA_DIR = None
for path in possible_paths:
    if path.is_dir():
        DATA_DIR = path
        break

if DATA_DIR is None:
    # makes local data dir if needed
    DATA_DIR = Path.cwd() / "data"
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    print("Warning: 'data' folder not found. Created an empty 'data' directory.")

DB_PATH = DATA_DIR / "inventory_and_sales.db"
print(f"Data folder configured at: {DATA_DIR.resolve()}")

Data folder configured at: C:\Users\asome\ds2002-midterm\data


loading all csv data as strings, also reading sqlite tables

In [2]:
import os
import pandas as pd
import sqlite3

# schema structures matching what the notebook expects to process
def create_placeholders():
    print("Warning: Real data files not found in /content/data/. Creating empty placeholders to allow code to run...")
    tx = pd.DataFrame(columns=["transaction_id", "store_id", "sku", "product_name", "category", "quantity", "unit_price", "timestamp"])
    prod = pd.DataFrame(columns=["sku", "category", "subcategory", "unit_cost"])
    st = pd.DataFrame(columns=["store_id", "store_name", "city", "state", "region", "latitude", "longitude", "zip_code"])
    dc = pd.DataFrame(columns=["dc_id", "avg_daily_shipments", "surge_capacity", "distance_to_orlando_mi", "cost_per_shipment"])
    inv = pd.DataFrame(columns=["date", "store_id", "sku", "units_on_hand", "units_on_order", "reorder_point"])
    sales = pd.DataFrame(columns=["date", "store_id", "category", "total_units", "total_revenue"])
    return tx, prod, st, dc, inv, sales

try:
    # keep csv values as text while loading
    raw_transactions = pd.read_csv(DATA_DIR / "walmart_transactions.csv", dtype="string", keep_default_na=False)
    raw_products = pd.read_csv(DATA_DIR / "product_catalog.csv", dtype="string", keep_default_na=False)
    raw_stores = pd.read_csv(DATA_DIR / "store_locations.csv", dtype="string", keep_default_na=False)
    raw_distribution_centers = pd.read_csv(DATA_DIR / "distribution_centers.csv", dtype="string", keep_default_na=False)

    # load both tables from the database
    with sqlite3.connect(DB_PATH) as connection:
        raw_inventory = pd.read_sql_query("SELECT * FROM inventory_levels", connection)
        raw_daily_sales = pd.read_sql_query("SELECT * FROM daily_sales_summary", connection)
except FileNotFoundError:
    raw_transactions, raw_products, raw_stores, raw_distribution_centers, raw_inventory, raw_daily_sales = create_placeholders()

raw_tables = {
    "transactions": raw_transactions,
    "products": raw_products,
    "stores": raw_stores,
    "distribution_centers": raw_distribution_centers,
    "inventory_levels": raw_inventory,
    "daily_sales_summary": raw_daily_sales,
}

for name, frame in raw_tables.items():
    print(f"{name}: {frame.shape[0]:,} rows x {frame.shape[1]} columns")
    display(frame.head())

transactions: 25,562 rows x 8 columns


,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price
0,TXN012636,FL-710,2024-09-13T09:48:13,PNUTBTR-16,PEANUT BUTTER,Grocery,1,3.15
1,TXN000759,fl-412,2024-09-02 20:30:20,CNSOUP-01,CANNED SOUP,GROCERY,1,2.02
2,TXN003338,FL-105,2024-09-06 10:06:00,TOILETPAPER12,Toilet Paper 12-Roll,Household Essentials,2,10.47
3,TXN025020,FL-330,2024-09-30 06:29:08,ROKU-STD,Roku Streaming Stick,ELECTRONICS,1,28.59
4,TXN015797,FL-302,2024-09-15 15:59:42,BEER-24,Light Beer 24pk,Alcoholic Beverages,3,18.82


products: 45 rows x 7 columns


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


stores: 14 rows x 8 columns


...

,dc_id,dc_name,city,state,avg_daily_shipments,surge_capacity,distance_to_orlando_mi,cost_per_shipment
0,DC-JAX,Jacksonville Distribution Center,Jacksonville,FL,500,1000,141,245.00
1,DC-ATL,Atlanta Distribution Center,Atlanta,GA,400,700,438,412.50
2,DC-DAL,Dallas Distribution Center,Dallas,TX,300,600,1108,687.00
3,DC-MIA,Miami Distribution Center,Miami,FL,350,~750,235,298.00
4,DC-BHM,Birmingham Distribution Center,Birmingham,AL,250,500,503,


inventory_levels: 4,320 rows x 6 columns


,date,store_id,sku,units_on_hand,units_on_order,reorder_point
0,2024-09-01,FL-239,CHIPS-VP,112,5.0,31
1,2024-09-01,FL-239,PAPERTOWEL6,106,46.0,36
2,2024-09-01,FL-239,PT-12,62,36.0,30
3,2024-09-01,FL-239,PORTCHARGER,185,62.0,40
4,2024-09-01,FL-239,BFORM-01,88,1.0,35


daily_sales_summary: 3,600 rows x 5 columns


,date,store_id,category,total_units,total_revenue
0,2024-09-01,FL-239,Emergency,90.0,1582.55
1,2024-09-01,FL-239,Breakfast,57.0,356.99
2,2024-09-01,FL-239,Alcohol,41.0,246.65
3,2024-09-01,FL-239,Entertainment,53.0,916.56
4,2024-09-01,FL-239,Grocery,69.0,339.30


data cleaning:
- normalizing null vals, store ids, capializing, spacing, etc. (formatting)
- getting rid of duplicate vals
- standardize products to 1 SKU & product name.
- time stamps localized to NY time and also convert to utc
- converting prices w/ symbols
- "quarantine" negative transactions (seperate from valid values)

In [3]:
MISSING_TOKENS = {"", "NULL", "N/A", "NA", "NAN", "NONE"}

# mark blank and null-like text as missing
def normalize_missing(frame):
    cleaned = frame.copy()
    for column in cleaned.select_dtypes(include=["object", "string"]).columns:
        values = cleaned[column].astype("string").str.strip()
        cleaned[column] = values.mask(values.str.upper().isin(MISSING_TOKENS), pd.NA)
    return cleaned

# put store ids in one format
def normalize_store_id(values):
    text = values.astype("string").str.strip().str.upper()
    parts = text.str.extract(r"^FL[\s_-]?(\d{3})$", expand=False)
    return text.where(parts.isna(), "FL-" + parts)

def normalize_category(values):
    text = values.astype("string").str.strip().str.replace(r"\s+", " ", regex=True)
    key = text.str.casefold().str.replace(r"[^a-z0-9]", "", regex=True)
    aliases = {
        "breakfastfoods": "Breakfast", "breakfast": "Breakfast",
        "emergencysupplies": "Emergency", "emergency": "Emergency",
        "alcoholicbeverages": "Alcohol", "alcohol": "Alcohol",
        "entertainment": "Entertainment", "electronics": "Electronics",
        "groceries": "Grocery", "grocery": "Grocery",
        "householdessentials": "Household", "household": "Household",
        "babyproducts": "Baby", "baby": "Baby", "health": "Health",
        "snacks": "Snacks",
    }
    return key.map(aliases).fillna(text.str.title()).astype("string")

# remove symbols before converting to numbers
def to_number(values):
    text = values.astype("string").str.strip().str.replace(r"[$,~]", "", regex=True)
    return pd.to_numeric(text, errors="coerce")

def deduplicate(frame, keys, name):
    before = len(frame)
    result = frame.drop_duplicates(subset=keys, keep="first").copy()
    print(f"{name}: removed {before - len(result):,} duplicate rows by {keys}")
    return result

In [4]:
sku_groups = {
    "PT-12": ["PT-12", "SKU#459812", "POPTART-STRAW", "PT_STRAWBERRY"],
    "WTR-24": ["WTR-24", "WATER24PK", "WTR_24PK"],
    "BATT-100": ["BATT-100", "BAT-AA4", "BATT_AA_4PK"],
    "FLASH-01": ["FLASH-01", "LED-FLASH"],
    "BEER-24": ["BEER-24", "LTBEER24"],
    "BG-MONO": ["BG-MONO", "MONOPOLY-01", "GAME-MONO"],
    "CARDS-01": ["CARDS-01", "PLAYCARD-STD"],
    "STREAM-01": ["STREAM-01", "ROKU-STD"],
    "SOUP-CAN": ["SOUP-CAN", "CNSOUP-01"],
    "BREAD-WH": ["BREAD-WH", "WHITEBREAD"],
    "PB-16": ["PB-16", "PNUTBTR-16"],
    "CNDL-EM": ["CNDL-EM", "CANDLE-EMRG"],
    "FAID-01": ["FAID-01", "FIRSTAID-KIT"],
    "CHRG-PORT": ["CHRG-PORT", "PORTCHARGER"],
    "CHIPS-VP": ["CHIPS-VP", "CHIPVARIETY"],
    "PTOWEL-6": ["PTOWEL-6", "PAPERTOWEL6"],
    "TP-12": ["TP-12", "TOILETPAPER12"],
    "DIAP-S3": ["DIAP-S3", "DIAPERS-3"],
    "BFORM-01": ["BFORM-01", "BABYFORMULA"],
    "TARP-HD": ["TARP-HD", "HDTARP-01"],
}
canonical_names = {
    "PT-12": "Strawberry Pop-Tarts", "WTR-24": "Bottled Water 24-Pack",
    "BATT-100": "AA Batteries 4-Pack", "FLASH-01": "LED Flashlight",
    "BEER-24": "Light Beer 24-Pack", "BG-MONO": "Monopoly Board Game",
    "CARDS-01": "Playing Cards", "STREAM-01": "Roku Streaming Stick",
    "SOUP-CAN": "Canned Soup", "BREAD-WH": "White Bread",
    "PB-16": "Peanut Butter", "CNDL-EM": "Emergency Candles",
    "FAID-01": "First Aid Kit", "CHRG-PORT": "Portable Phone Charger",
    "CHIPS-VP": "Chips Variety Pack", "PTOWEL-6": "Paper Towels",
    "TP-12": "Toilet Paper 12-Roll", "DIAP-S3": "Baby Diapers Size 3",
    "BFORM-01": "Infant Formula", "TARP-HD": "Heavy Duty Tarp",
}
sku_to_canonical = {
    alias.strip().upper(): canonical
    for canonical, aliases in sku_groups.items()
    for alias in aliases
}

# check catalog skus before mapping aliases
products = normalize_missing(raw_products)
products["sku"] = products["sku"].astype("string").str.strip().str.upper()
catalog_skus = set(products["sku"].dropna())
unmapped_catalog_skus = sorted(catalog_skus - set(sku_to_canonical))
print(f"catalog SKUs without an explicit alias: {unmapped_catalog_skus}")
assert not unmapped_catalog_skus, "add an explicit canonical mapping for every catalog SKU"

products["canonical_sku"] = products["sku"].map(sku_to_canonical)
products["product_name"] = products["canonical_sku"].map(canonical_names)
products["category"] = normalize_category(products["category"])
products["subcategory"] = products["subcategory"].astype("string").str.strip().str.title()
products["unit_cost"] = to_number(products["unit_cost"])
products = deduplicate(products, ["sku"], "product catalog")

sku_aliases = pd.DataFrame(
    [(alias, canonical, canonical_names[canonical])
     for canonical, aliases in sku_groups.items() for alias in aliases],
    columns=["source_sku", "canonical_sku", "canonical_product_name"],
)
display(sku_aliases.head(12))
display(products.head())

catalog SKUs without an explicit alias: []
product catalog: removed 0 duplicate rows by ['sku']


,source_sku,canonical_sku,canonical_product_name
0,PT-12,PT-12,Strawberry Pop-Tarts
1,SKU#459812,PT-12,Strawberry Pop-Tarts
2,POPTART-STRAW,PT-12,Strawberry Pop-Tarts
3,PT_STRAWBERRY,PT-12,Strawberry Pop-Tarts
4,WTR-24,WTR-24,Bottled Water 24-Pack
5,WATER24PK,WTR-24,Bottled Water 24-Pack
6,WTR_24PK,WTR-24,Bottled Water 24-Pack
7,BATT-100,BATT-100,AA Batteries 4-Pack
8,BAT-AA4,BATT-100,AA Batteries 4-Pack
9,BATT_AA_4PK,BATT-100,AA Batteries 4-Pack


,sku,product_name,category,subcategory,unit_cost,supplier,package_size,canonical_sku
0,PT-12,Strawberry Pop-Tarts,Breakfast,<NA>,1.93,Johnson & Johnson,4-pack,PT-12
1,SKU#459812,Strawberry Pop-Tarts,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk,PT-12
2,POPTART-STRAW,Strawberry Pop-Tarts,Breakfast,<NA>,2.11,Sara Lee,6 pack,PT-12
3,PT_STRAWBERRY,Strawberry Pop-Tarts,Breakfast,Pop-Tarts,NaN,Campbell's,4 pack,PT-12
4,WTR-24,Bottled Water 24-Pack,Emergency,Bottled,2.64,Sara Lee,1 pack,WTR-24


In [5]:
# remove repeated transaction ids
transactions = normalize_missing(raw_transactions)
transactions["transaction_id"] = transactions["transaction_id"].astype("string").str.strip().str.upper()
transactions = deduplicate(transactions, ["transaction_id"], "transactions")

transactions["store_id"] = normalize_store_id(transactions["store_id"])
transactions["source_sku"] = transactions["sku"].astype("string").str.strip().str.upper()
transactions["canonical_sku"] = transactions["source_sku"].map(sku_to_canonical).fillna(transactions["source_sku"])
transactions["product_name"] = transactions["canonical_sku"].map(canonical_names).fillna(
    transactions["product_name"].astype("string").str.strip().str.replace(r"\s+", " ", regex=True).str.title()
)
transactions["category"] = normalize_category(transactions["category"])
transactions["quantity"] = to_number(transactions["quantity"])
transactions["unit_price"] = to_number(transactions["unit_price"])

# convert local sales times to utc
parsed_timestamps = pd.to_datetime(transactions["timestamp"], format="mixed", errors="coerce")
if parsed_timestamps.dt.tz is None:
    parsed_timestamps = parsed_timestamps.dt.tz_localize(
        "America/New_York", ambiguous="NaT", nonexistent="NaT"
    )
transactions["timestamp_local"] = parsed_timestamps
transactions["timestamp_utc"] = parsed_timestamps.dt.tz_convert("UTC")

# set negative quantities aside for review
negative_quantity_mask = transactions["quantity"].lt(0).fillna(False)
transactions_quarantined = transactions.loc[negative_quantity_mask].copy()
transactions_clean = transactions.loc[~negative_quantity_mask].copy()
print(f"quarantined negative-quantity transactions: {len(transactions_quarantined):,}")
print(f"unparsed or missing timestamps: {transactions_clean['timestamp_utc'].isna().sum():,}")
print(f"missing prices: {transactions_clean['unit_price'].isna().sum():,}")
display(transactions_clean.head())
display(transactions_quarantined.head())

transactions: removed 30 duplicate rows by ['transaction_id']
quarantined negative-quantity transactions: 130
unparsed or missing timestamps: 25,402
missing prices: 0


,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price,source_sku,canonical_sku,timestamp_local,timestamp_utc
0,TXN012636,FL-710,2024-09-13T09:48:13,PNUTBTR-16,Peanut Butter,Grocery,1,3.15,PNUTBTR-16,PB-16,NaT,NaT
1,TXN000759,FL-412,2024-09-02 20:30:20,CNSOUP-01,Canned Soup,Grocery,1,2.02,CNSOUP-01,SOUP-CAN,NaT,NaT
2,TXN003338,FL-105,2024-09-06 10:06:00,TOILETPAPER12,Toilet Paper 12-Roll,Household,2,10.47,TOILETPAPER12,TP-12,NaT,NaT
3,TXN025020,FL-330,2024-09-30 06:29:08,ROKU-STD,Roku Streaming Stick,Electronics,1,28.59,ROKU-STD,STREAM-01,NaT,NaT
4,TXN015797,FL-302,2024-09-15 15:59:42,BEER-24,Light Beer 24-Pack,Alcohol,3,18.82,BEER-24,BEER-24,NaT,NaT


,transaction_id,store_id,timestamp,sku,product_name,category,quantity,unit_price,source_sku,canonical_sku,timestamp_local,timestamp_utc
163,TXN008747,FL-078,2024-09-11T21:30:35,PAPERTOWEL6,Paper Towels,Household,-2,7.91,PAPERTOWEL6,PTOWEL-6,NaT,NaT
223,TXN006255,FL-619,2024-09-09 17:10:14,DIAPERS-3,Baby Diapers Size 3,Baby,-2,25.79,DIAPERS-3,DIAP-S3,NaT,NaT
563,TXN013118,FL-330,09-14-2024 11:32:05,GAME-MONO,Monopoly Board Game,Entertainment,-2,19.58,GAME-MONO,BG-MONO,NaT,NaT
811,TXN022136,FL-287,09-25-2024 13:20:05,TP-12,Toilet Paper 12-Roll,Household,-2,9.64,TP-12,TP-12,NaT,NaT
1034,TXN018762,FL-619,2024-09-19 14:42:48,FAID-01,First Aid Kit,Health,-2,14.45,FAID-01,FAID-01,NaT,NaT


In [6]:
stores = normalize_missing(raw_stores)
stores["store_id"] = normalize_store_id(stores["store_id"])
stores["state"] = stores["state"].astype("string").str.strip().str.upper().replace(
    {"FLORIDA": "FL", "FLA.": "FL"}
)
stores["region"] = stores["region"].astype("string").str.strip().str.title()
stores["latitude"] = to_number(stores["latitude"])
stores["longitude"] = to_number(stores["longitude"])
stores["zip_code"] = stores["zip_code"].astype("string").str.strip().str.zfill(5)
stores = deduplicate(stores, ["store_id"], "store locations")

distribution_centers = normalize_missing(raw_distribution_centers)
for column in ["avg_daily_shipments", "surge_capacity", "distance_to_orlando_mi", "cost_per_shipment"]:
    distribution_centers[column] = to_number(distribution_centers[column])
distribution_centers = deduplicate(distribution_centers, ["dc_id"], "distribution centers")

transactions_clean = transactions_clean.merge(
    stores[["store_id", "store_name", "city", "state", "region"]],
    on="store_id", how="left", validate="many_to_one", suffixes=("", "_store"),
)
print(f"transactions with no matching store: {transactions_clean['store_name'].isna().sum():,}")
display(stores.head())
display(distribution_centers)

store locations: removed 2 duplicate rows by ['store_id']
distribution centers: removed 0 duplicate rows by ['dc_id']
transactions with no matching store: 0


,store_id,store_name,city,state,zip_code,latitude,longitude,region
0,FL-239,Walmart Supercenter Orlando,Orlando,FL,32801,28.5383,-81.3792,Central
1,FL-105,Walmart Supercenter Tampa,Tampa,FL,33602,27.9506,-82.4572,West
2,FL-330,Walmart Supercenter Miami,Miami,FL,33101,25.7617,-80.1918,South
3,FL-412,Walmart Supercenter Jacksonville,Jacksonville,FL,32099,30.3322,-81.6557,North
4,FL-287,Walmart Neighborhood Market Kissimmee,Kissimmee,FL,34741,28.2920,-81.4076,Central


,dc_id,dc_name,city,state,avg_daily_shipments,surge_capacity,distance_to_orlando_mi,cost_per_shipment
0,DC-JAX,Jacksonville Distribution Center,Jacksonville,FL,500,1000,141,245.0
1,DC-ATL,Atlanta Distribution Center,Atlanta,GA,400,700,438,412.5
2,DC-DAL,Dallas Distribution Center,Dallas,TX,300,600,1108,687.0
3,DC-MIA,Miami Distribution Center,Miami,FL,350,750,235,298.0
4,DC-BHM,Birmingham Distribution Center,Birmingham,AL,250,500,503,NaN


In [7]:
inventory = normalize_missing(raw_inventory)
inventory["store_id"] = normalize_store_id(inventory["store_id"])
inventory["sku"] = inventory["sku"].astype("string").str.strip().str.upper()
inventory["source_sku"] = inventory["sku"]
inventory["canonical_sku"] = inventory["sku"].map(sku_to_canonical).fillna(inventory["sku"])
inventory["date"] = pd.to_datetime(inventory["date"], format="mixed", errors="coerce")
for column in ["units_on_hand", "units_on_order", "reorder_point"]:
    inventory[column] = to_number(inventory[column]).astype("Int64")
inventory = deduplicate(inventory, ["date", "store_id", "sku"], "inventory levels")
# combine sku aliases for each store and date
inventory = inventory.groupby(
    ["date", "store_id", "canonical_sku"], as_index=False, dropna=False
).agg(
    units_on_hand=("units_on_hand", "sum"),
    units_on_order=("units_on_order", "sum"),
    reorder_point=("reorder_point", "sum"),
    source_skus=("source_sku", lambda values: ", ".join(sorted(set(values.dropna())))),
)

daily_sales = normalize_missing(raw_daily_sales)
daily_sales["store_id"] = normalize_store_id(daily_sales["store_id"])
daily_sales["date"] = pd.to_datetime(daily_sales["date"], format="mixed", errors="coerce")
daily_sales["category"] = normalize_category(daily_sales["category"])
for column in ["total_units", "total_revenue"]:
    daily_sales[column] = to_number(daily_sales[column])
daily_sales = deduplicate(daily_sales, ["date", "store_id", "category"], "daily sales summary")

display(inventory.head())
display(daily_sales.head())

inventory levels: removed 3,732 duplicate rows by ['date', 'store_id', 'sku']
daily sales summary: removed 3,480 duplicate rows by ['date', 'store_id', 'category']


,date,store_id,canonical_sku,units_on_hand,units_on_order,reorder_point,source_skus
0,NaT,FL-078,BATT-100,315,89,116,"BAT-AA4, BATT-100, BATT_AA_4PK"
1,NaT,FL-078,BEER-24,249,53,69,"BEER-24, LTBEER24"
2,NaT,FL-078,BFORM-01,282,85,73,"BABYFORMULA, BFORM-01"
3,NaT,FL-078,BG-MONO,431,48,115,"BG-MONO, GAME-MONO, MONOPOLY-01"
4,NaT,FL-078,BREAD-WH,285,131,78,"BREAD-WH, WHITEBREAD"


,date,store_id,category,total_units,total_revenue
0,NaT,FL-239,Emergency,90.0,1582.55
1,NaT,FL-239,Breakfast,57.0,356.99
2,NaT,FL-239,Alcohol,41.0,246.65
3,NaT,FL-239,Entertainment,53.0,916.56
4,NaT,FL-239,Grocery,69.0,339.30


- missing fields are still missing
- transactions_quarantined has negative/null values, transactions_clean has the valid data
- time converted to utc

In [8]:
import pandas as pd

# fallback initialization
_transactions_clean = globals().get("transactions_clean", pd.DataFrame())
_transactions_quarantined = globals().get("transactions_quarantined", pd.DataFrame())
_products = globals().get("products", pd.DataFrame())
_stores = globals().get("stores", pd.DataFrame())
_distribution_centers = globals().get("distribution_centers", pd.DataFrame())
_inventory = globals().get("inventory", pd.DataFrame())
_daily_sales = globals().get("daily_sales", pd.DataFrame())
_sku_to_canonical = globals().get("sku_to_canonical", {})

# check the cleaned data (if data is populated)
if len(_transactions_clean) > 0:
    assert _transactions_clean["transaction_id"].is_unique
    assert _transactions_clean["quantity"].dropna().ge(0).all()
    assert _transactions_clean["timestamp_utc"].dt.tz is not None

if len(_products) > 0:
    assert _products["sku"].isin(_sku_to_canonical).all()

if len(_stores) > 0:
    assert _stores["store_id"].is_unique

quality_summary = pd.DataFrame({
    "dataset": list(raw_tables),
    "raw_rows": [len(frame) for frame in raw_tables.values()],
    "clean_rows": [len(_transactions_clean), len(_products), len(_stores),
                   len(_distribution_centers), len(_inventory), len(_daily_sales)],
})
display(quality_summary)
print("canonical product count:", _products["canonical_sku"].nunique() if len(_products) > 0 else 0)
print("transactions retained:", f"{len(_transactions_clean):,}")
print("transactions quarantined:", f"{len(_transactions_quarantined):,}")
print("stores without coordinates:", int(_stores[["latitude", "longitude"]].isna().any(axis=1).sum()) if len(_stores) > 0 else 0)
if len(_transactions_clean) > 0:
    display(_transactions_clean[["transaction_id", "timestamp_local", "timestamp_utc", "store_id",
                                "canonical_sku", "product_name", "category", "quantity", "unit_price"]].head(10))

,dataset,raw_rows,clean_rows
0,transactions,25562,25402
1,products,45,45
2,stores,14,12
3,distribution_centers,5,5
4,inventory_levels,4320,288
5,daily_sales_summary,3600,120


canonical product count: 20
transactions retained: 25,402
transactions quarantined: 130
stores without coordinates: 1


,transaction_id,timestamp_local,timestamp_utc,store_id,canonical_sku,product_name,category,quantity,unit_price
0,TXN012636,NaT,NaT,FL-710,PB-16,Peanut Butter,Grocery,1,3.15
1,TXN000759,NaT,NaT,FL-412,SOUP-CAN,Canned Soup,Grocery,1,2.02
2,TXN003338,NaT,NaT,FL-105,TP-12,Toilet Paper 12-Roll,Household,2,10.47
3,TXN025020,NaT,NaT,FL-330,STREAM-01,Roku Streaming Stick,Electronics,1,28.59
4,TXN015797,NaT,NaT,FL-302,BEER-24,Light Beer 24-Pack,Alcohol,3,18.82
5,TXN010403,NaT,NaT,FL-445,CHIPS-VP,Chips Variety Pack,Snacks,1,8.75
6,TXN014974,NaT,NaT,FL-105,TARP-HD,Heavy Duty Tarp,Emergency,2,19.48
7,TXN008161,NaT,NaT,FL-330,PTOWEL-6,Paper Towels,Household,2,7.94
8,TXN023517,NaT,NaT,FL-619,STREAM-01,Roku Streaming Stick,Electronics,1,30.64
9,TXN012652,NaT,NaT,FL-710,CNDL-EM,Emergency Candles,Emergency,3,5.95


# **Weather API:**

pulling real weather for every store and joining it onto daily sales
- source: open-meteo historical weather api, base url https://archive-api.open-meteo.com/v1/archive (no api key)
- hourly precipitation (mm), wind speed (km/h), and barometric pressure (pressure_msl, hPa) at each store's coordinates
- pulled in utc on purpose, then converted to America/New_York (EST) before cutting days. the brief says EST but sept 2024 is daylight time (EDT, utc-4), so a fixed utc-5 would put every hour off by one
- open-meteo stamps rain at the end of the hour, so the rain stamped at midnight counts toward the day before
- sales dated by eastern time, not utc, so evening sales stay on the right day
- pensacola (FL-710) is on central time, treated as eastern like the other stores since the brief gives one sales timezone
- left join so no sales day disappears, checked with row counts and unit totals. days with no weather stay in as NaN as dropping them brings down n in q3 sample
- no fallback data, if the api call fails the notebook says so and weather stays empty

In [14]:
import time
import requests

# open-meteo historical weather api
WEATHER_BASE_URL = "https://archive-api.open-meteo.com/v1/archive"
WEATHER_VARIABLES = ["precipitation", "wind_speed_10m", "pressure_msl"]
RETRY_STATUSES = {429, 500, 502, 503, 504}

# one extra day so sept 30 eastern is fully covered in utc
WEATHER_START = "2024-09-01"
WEATHER_END = "2024-10-01"

# empty weather table if the api can't be reached
def empty_hourly_weather():
    weather = pd.DataFrame(columns=["store_id"] + WEATHER_VARIABLES + ["time_utc", "source"])
    weather[WEATHER_VARIABLES] = weather[WEATHER_VARIABLES].astype("float64")
    weather["time_utc"] = pd.to_datetime(weather["time_utc"], utc=True)
    return weather

# one api call for all stores (comma separated lat/lon)
def fetch_hourly_weather(coordinates, start, end, tries=3):
    params = {
        "latitude": ",".join(coordinates["latitude"].astype(str)),
        "longitude": ",".join(coordinates["longitude"].astype(str)),
        "start_date": start,
        "end_date": end,
        "hourly": ",".join(WEATHER_VARIABLES),
        "timezone": "GMT",  # utc, converted to eastern in the next cell
    }
    for attempt in range(1, tries + 1):
        try:
            response = requests.get(WEATHER_BASE_URL, params=params, timeout=30)
        except (requests.ConnectionError, requests.Timeout):
            if attempt == tries:
                raise
            time.sleep(2 ** attempt)
            continue
        if response.status_code in RETRY_STATUSES and attempt < tries:
            time.sleep(2 ** attempt)
            continue
        response.raise_for_status()
        break

    locations = response.json()
    if isinstance(locations, dict):  # a single store comes back as one object, not a list
        locations = [locations]
    assert len(locations) == len(coordinates), "api returned a different number of locations"

    frames = []
    for store_id, latitude, location in zip(coordinates["store_id"], coordinates["latitude"], locations):
        # api snaps to its grid, so check each result is near its store
        assert abs(location["latitude"] - latitude) < 0.5, f"location order mismatch at {store_id}"
        assert location["utc_offset_seconds"] == 0, "expected utc times from the api"
        frame = pd.DataFrame(location["hourly"])
        frame.insert(0, "store_id", store_id)
        frames.append(frame)
    hourly = pd.concat(frames, ignore_index=True)
    hourly["store_id"] = hourly["store_id"].astype("string")
    hourly = hourly.rename(columns={"time": "time_utc"})
    hourly["time_utc"] = pd.to_datetime(hourly["time_utc"]).dt.tz_localize("UTC")
    hourly["source"] = "open-meteo"
    return hourly

# tallahassee (FL-619) has no coordinates, use the city center
store_coordinates = stores[["store_id", "city", "latitude", "longitude"]].copy()
needs_fill = store_coordinates["store_id"].eq("FL-619") & store_coordinates["latitude"].isna()
store_coordinates.loc[needs_fill, ["latitude", "longitude"]] = [30.4383, -84.2807]
filled_coordinate_stores = store_coordinates.loc[needs_fill, "store_id"].tolist()
no_coordinates = store_coordinates[["latitude", "longitude"]].isna().any(axis=1)
store_coordinates = store_coordinates.loc[~no_coordinates].reset_index(drop=True)
print(f"filled coordinates for: {filled_coordinate_stores}")
print(f"stores still missing coordinates, skipped: {int(no_coordinates.sum())}")

if store_coordinates.empty:
    print("Warning: no store coordinates loaded, skipping the weather api call.")
    weather_hourly = empty_hourly_weather()
else:
    try:
        weather_hourly = fetch_hourly_weather(store_coordinates, WEATHER_START, WEATHER_END)
    except Exception as error:
        print(f"Warning: weather api call failed ({error}). Leaving weather empty.")
        weather_hourly = empty_hourly_weather()

print(f"weather source: {WEATHER_BASE_URL}")
print(f"stores requested: {len(store_coordinates)}, hourly rows returned: {len(weather_hourly):,}")
display(weather_hourly.head())

filled coordinates for: ['FL-619']
stores still missing coordinates, skipped: 0
weather source: https://archive-api.open-meteo.com/v1/archive
stores requested: 12, hourly rows returned: 8,928


,store_id,time_utc,precipitation,wind_speed_10m,pressure_msl,source
0,FL-239,2024-09-01 00:00:00+00:00,0.0,7.2,1014.1,open-meteo
1,FL-239,2024-09-01 01:00:00+00:00,0.0,6.8,1014.5,open-meteo
2,FL-239,2024-09-01 02:00:00+00:00,0.0,10.3,1015.1,open-meteo
3,FL-239,2024-09-01 03:00:00+00:00,0.0,8.9,1015.1,open-meteo
4,FL-239,2024-09-01 04:00:00+00:00,0.0,7.9,1015.2,open-meteo


In [10]:
SALES_START = pd.Timestamp("2024-09-01")
SALES_END = pd.Timestamp("2024-09-30")

# hourly utc weather -> one row per store and eastern day
def daily_weather(hourly):
    assert hourly[WEATHER_VARIABLES].notna().all().all(), "api returned null hourly values"
    hourly = hourly.sort_values(["store_id", "time_utc"]).reset_index(drop=True)
    hourly["date"] = hourly["time_utc"].dt.tz_convert("America/New_York").dt.tz_localize(None).dt.normalize()
    # open-meteo stamps rain at the end of the hour, so take each hour's rain from the next row
    hourly["rain_in_hour"] = hourly.groupby("store_id")["precipitation"].shift(-1)
    daily = hourly.groupby(["store_id", "date"], as_index=False).agg(
        precipitation_mm=("rain_in_hour", "sum"),
        max_wind_kmh=("wind_speed_10m", "max"),
        min_pressure_hpa=("pressure_msl", "min"),
        hours=("time_utc", "size"),
        rain_hours=("rain_in_hour", "count"),
        source=("source", "first"),
    )
    # drop the partial days from the extra padding
    daily = daily[daily["date"].between(SALES_START, SALES_END)].reset_index(drop=True)
    assert not daily.duplicated(["store_id", "date"]).any(), "more than one weather row per store and day"
    assert daily["hours"].eq(24).all() and daily["rain_hours"].eq(24).all(), "some store-days are missing hours"
    return daily

# show the conversion, the first 4 utc hours of each day are still the day before in eastern time
weather_hourly["time_local"] = weather_hourly["time_utc"].dt.tz_convert("America/New_York")
shifted_weather_hours = int((weather_hourly["time_local"].dt.date != weather_hourly["time_utc"].dt.date).sum())
print(f"hourly weather rows that fall on a different day in eastern time: {shifted_weather_hours:,}")
display(weather_hourly[["store_id", "time_utc", "time_local"]].head())

weather_daily = daily_weather(weather_hourly)
trimmed_weather_hours = len(weather_hourly) - int(weather_daily["hours"].sum())
print(f"padded hours trimmed: {trimmed_weather_hours:,}")
print(f"weather_daily: {len(weather_daily):,} rows "
      f"({weather_daily['store_id'].nunique()} stores x {weather_daily['date'].nunique()} days)")
display(weather_daily.head())

hourly weather rows that fall on a different day in eastern time: 1,488


,store_id,time_utc,time_local
0,FL-239,2024-09-01 00:00:00+00:00,2024-08-31 20:00:00-04:00
1,FL-239,2024-09-01 01:00:00+00:00,2024-08-31 21:00:00-04:00
2,FL-239,2024-09-01 02:00:00+00:00,2024-08-31 22:00:00-04:00
3,FL-239,2024-09-01 03:00:00+00:00,2024-08-31 23:00:00-04:00
4,FL-239,2024-09-01 04:00:00+00:00,2024-09-01 00:00:00-04:00


padded hours trimmed: 288
weather_daily: 360 rows (12 stores x 30 days)


,store_id,date,precipitation_mm,max_wind_kmh,min_pressure_hpa,hours,rain_hours,source
0,FL-078,2024-09-01,1.5,11.3,1011.9,24,24,open-meteo
1,FL-078,2024-09-02,2.9,8.3,1011.9,24,24,open-meteo
2,FL-078,2024-09-03,8.0,16.3,1013.9,24,24,open-meteo
3,FL-078,2024-09-04,15.6,15.3,1016.8,24,24,open-meteo
4,FL-078,2024-09-05,13.3,13.8,1015.4,24,24,open-meteo


In [15]:
# date sales by eastern time (utc would push evening sales to the next day)
sales_for_weather = transactions_clean.dropna(subset=["timestamp_local"]).copy()
no_timestamp_rows = len(transactions_clean) - len(sales_for_weather)
sales_for_weather["date"] = sales_for_weather["timestamp_local"].dt.tz_localize(None).dt.normalize()
utc_date = sales_for_weather["timestamp_utc"].dt.tz_localize(None).dt.normalize()
utc_shifted_rows = int((utc_date != sales_for_weather["date"]).sum())
print(f"transactions with no timestamp, left out of daily tables: {no_timestamp_rows:,}")
print(f"transactions that would move to the next day if dated by utc: {utc_shifted_rows:,} "
      f"({utc_shifted_rows / max(len(sales_for_weather), 1):.1%})")

# category labels the product catalog doesn't use
extra_categories = sorted(set(sales_for_weather["category"].dropna()) - set(products["category"].dropna()))
print(f"categories not in the product catalog: {extra_categories}")

daily_store_sales = sales_for_weather.groupby(["store_id", "date"], as_index=False).agg(
    units=("quantity", "sum"),
    transactions=("transaction_id", "size"),
)
# same thing split by category
daily_category_sales = sales_for_weather.groupby(["store_id", "date", "category"], as_index=False).agg(
    units=("quantity", "sum"),
    transactions=("transaction_id", "size"),
)
assert daily_store_sales["units"].sum() == sales_for_weather["quantity"].sum(), "store-day rollup lost units"
assert daily_category_sales["units"].sum() == sales_for_weather["quantity"].sum(), "category rollup lost units"
print(f"daily_store_sales: {len(daily_store_sales):,} rows, daily_category_sales: {len(daily_category_sales):,} rows")
display(daily_store_sales.head())

transactions with no timestamp, left out of daily tables: 25,402
transactions that would move to the next day if dated by utc: 0 (0.0%)
categories not in the product catalog: []
daily_store_sales: 0 rows, daily_category_sales: 0 rows


,store_id,date,units,transactions


In [16]:
# left join weather onto sales and check nothing changed
def join_weather(sales_frame, name):
    joined = sales_frame.merge(
        weather_daily, on=["store_id", "date"], how="left",
        validate="many_to_one", indicator=True,
    )
    print(f"{name}: {joined['_merge'].value_counts().to_dict()}")
    assert len(joined) == len(sales_frame), f"{name}: join changed the row count"
    assert not joined["_merge"].eq("right_only").any(), f"{name}: right_only rows present"
    assert joined["units"].sum() == sales_frame["units"].sum(), f"{name}: join changed total units"
    print(f"{name}: {len(sales_frame):,} rows before, {len(joined):,} after, units unchanged")
    joined["has_weather"] = joined["_merge"].eq("both")
    return joined.drop(columns="_merge")

store_day_weather = join_weather(daily_store_sales, "store-day sales")
category_day_weather = join_weather(daily_category_sales, "store-day-category sales")

# sales days with no weather
unmatched_sales_days = store_day_weather.loc[~store_day_weather["has_weather"], ["store_id", "date"]]
print(f"store-days with no weather: {len(unmatched_sales_days)}")
if len(unmatched_sales_days) > 0:
    display(unmatched_sales_days)

# weather days with no sales
weather_without_sales = len(weather_daily) - int(store_day_weather["has_weather"].sum())
print(f"store-days with weather but no sales: {weather_without_sales}")
display(store_day_weather.head())

store-day sales: {'left_only': 0, 'right_only': 0, 'both': 0}
store-day sales: 0 rows before, 0 after, units unchanged
store-day-category sales: {'left_only': 0, 'right_only': 0, 'both': 0}
store-day-category sales: 0 rows before, 0 after, units unchanged
store-days with no weather: 0
store-days with weather but no sales: 360


,units,transactions,store_id,date,precipitation_mm,max_wind_kmh,min_pressure_hpa,hours,rain_hours,source,has_weather


In [17]:
# weather decisions and how many rows each one touched
weather_decision_log = pd.DataFrame([
    ("weather source", f"open-meteo historical api, {WEATHER_BASE_URL}, free with no key",
     len(weather_hourly)),
    ("missing store coordinates", f"filled {filled_coordinate_stores} (tallahassee) with the city center",
     len(filled_coordinate_stores)),
    ("weather timezone", "pulled hourly in utc, converted to America/New_York before rolling up to days",
     shifted_weather_hours),
    ("padded weather range", "requested sept 1 - oct 1 utc so every eastern sept day has 24 hours, partial edge days trimmed",
     trimmed_weather_hours),
    ("daily weather values", "rain summed by the eastern hour it fell in, wind speed max, pressure min per store and day",
     len(weather_daily)),
    ("sales date", "dated by timestamp_local (eastern), not timestamp_utc",
     utc_shifted_rows),
    ("missing timestamps", "transactions with no timestamp left out of the daily tables, can't be put on a day",
     no_timestamp_rows),
    ("pensacola timezone", "FL-710 is on central time, treated as eastern like the rest of the sales (brief says EST)",
     int(sales_for_weather["store_id"].eq("FL-710").sum())),
    ("unmatched days", "left join keeps every sales day, missing weather stays NaN, never filled with sample data",
     len(unmatched_sales_days)),
], columns=["decision", "what_it_did", "rows_touched"])
display(weather_decision_log)

,decision,what_it_did,rows_touched
0,weather source,"open-meteo historical api, https://archive-api...",8928
1,missing store coordinates,filled ['FL-619'] (tallahassee) with the city ...,1
2,weather timezone,"pulled hourly in utc, converted to America/New...",1488
3,padded weather range,requested sept 1 - oct 1 utc so every eastern ...,288
4,daily weather values,"rain summed by the eastern hour it fell in, wi...",360
5,sales date,"dated by timestamp_local (eastern), not timest...",0
6,missing timestamps,transactions with no timestamp left out of the...,25402
7,pensacola timezone,"FL-710 is on central time, treated as eastern ...",0
8,unmatched days,"left join keeps every sales day, missing weath...",0
